# PROJECT: MietRecht-AI-Local-Legal-Intelligence-for-Tenants

* Lease Analysis Assistant

Here is the main file that you will run. It pulls together your logic and the Gradio UI.

In [ ]:
import re
import gradio as gr
from llama_index.core import Settings, SimpleDirectoryReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.readers.file import PDFReader

# --- 1. CONFIGURATION ---

def configure_settings():
    Settings.llm = Ollama(model="llama3.2:1b", request_timeout=300.0, temperature=0.1)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )

# --- 2. LOGIC MODULES ---
def detect_clause_type(text):
    keywords = {
        "Kündigung (Termination)": ["kündigung", "frist", "termination", "notice"],
        "Schönheitsreparaturen (Repairs)": ["renovier", "maler", "repairs", "schönheit"],
        "Kaution (Deposit)": ["kaution", "sicherheit", "deposit"],
        "Nebenkosten (Utilities)": ["betriebskosten", "utility", "nebenkosten"]
    }
    for label, keys in keywords.items():
        if any(k in text.lower() for k in keys): return label
    return "Allgemeine Klausel / General Clause"

def process_lease(file_obj):
    if file_obj is None: return "Please upload a PDF."
    try:
        loader = PDFReader()
        documents = loader.load_data(file=file_obj.name)
        full_text = "\n".join([d.text for d in documents])

        sections = re.split(r'(\n§\s*\d+)', full_text)
        final_output = "# 📑 Section-by-Section Analysis\n\n"

        count = 0
        for i in range(1, len(sections), 2):
            if count >= 8: break
            clause_text = sections[i] + (sections[i+1] if i+1 < len(sections) else "")
            c_type = detect_clause_type(clause_text)

            prompt = f"System: You are a German legal expert. Analyze this clause. 1. Summarize in EN/DE. 2. Give a risk score 0-10. Clause: {clause_text}"
            response = Settings.llm.complete(prompt)
            final_output += f"### {c_type}\n{response.text}\n\n---\n"
            count += 1
        return final_output
    except Exception as e:
        return f"Error: {str(e)}"

def chat_logic(message, history):
    try:
        full_prompt = "You are a legal assistant specialized in German Mietrecht.\n"
        for user_msg, bot_msg in history:
            full_prompt += f"User: {user_msg}\nAssistant: {bot_msg}\n"
        full_prompt += f"User: {message}\nAssistant:"
        resp = Settings.llm.complete(full_prompt)
        return str(resp)
    except Exception as e:
        return f"Ollama Error: {e}"

# --- 3. UI LAYOUT ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏠 MietRecht AI - German Lease Assistant")

    with gr.Tab("📄 Lease Document Analysis"):
        file_input = gr.File(label="Upload PDF")
        analyze_btn = gr.Button("Analyze Clause by Clause", variant="primary")
        out_box = gr.Markdown(value="*Analysis will appear here...*")
        analyze_btn.click(fn=process_lease, inputs=file_input, outputs=out_box, show_progress="hidden")

    with gr.Tab("💬 General Q&A"):
        gr.ChatInterface(fn=chat_logic)

if __name__ == "__main__":
    demo.launch(share=True)